# 07 — Confluency evidence v3: the swapped replication + a fine-tuned MSC model

**What this runs (GPU only; scoring happens later on the Mac, once):**

- **Part A — the replication (mCellSeg, halves swapped).** Pre-registered in
  `results/confluency_mcellseg_swap.md`, committed before this notebook was written. Three fine-tuning runs on
  the 90 images that were the sealed test (two cross-fit folds + all 90), then their maps: out-of-fold for
  those 90, final-model for the other 79 (the new test images). About 1 h on A100.
- **Part B — a fine-tuned model for the MSC setup** (20 images from all three donor populations, fixed in the
  pre-registration). About 5 min. A measurement until the owner decides; its weights go to Drive separately.

**I/O rule.** Nothing is read through the Drive mount: the bundle zip is copied to local disk and unzipped;
mCellSeg is downloaded from Zenodo straight to local disk (MD5-checked). Setup checks that Colab runs the
same Cellpose-SAM weights as the product and the pre-registered files byte for byte.

**Before running:** upload `confluency_v3_bundle.zip` to `MyDrive/cultureqc/staged/`.

**Runtime:** A100 with **High-RAM** if offered (training holds 90 large images in memory). If the session
drops, re-run Setup and then only the part that did not finish.

**Bring back:** `MyDrive/cultureqc/staged/confluency_v3_results.zip` (a few MB). Leave
`msc_all_n20_s0.weights` (about 1 GB) on Drive; I'll ask for it only if you approve the MSC profile.

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, shutil, time, subprocess, json, hashlib, zipfile
DRIVE_ROOT = '/content/drive/MyDrive/cultureqc'
STAGE_DIR = f'{DRIVE_ROOT}/staged'
BUNDLE = f'{STAGE_DIR}/confluency_v3_bundle.zip'
RESULTS_ZIP = f'{STAGE_DIR}/confluency_v3_results.zip'
EXPECTED_COMMIT = '35cbed5'        # the pre-registration commit the bundle was made from
REPO = '/content/cultureqc'
assert os.path.exists(BUNDLE), f'upload the bundle first: {BUNDLE}'
print(f'{BUNDLE}: {os.path.getsize(BUNDLE)/1e6:.0f} MB')

In [ ]:
t0 = time.time()
shutil.rmtree(REPO, ignore_errors=True)
shutil.copy(BUNDLE, '/content/bundle.zip')                 # one sequential read from Drive
!unzip -q -o /content/bundle.zip -d $REPO
os.remove('/content/bundle.zip')
%cd $REPO
commit = open('COMMIT').read().strip()
print('bundle commit', commit, f'— unpacked in {time.time()-t0:.0f} s')
assert commit.startswith(EXPECTED_COMMIT), (commit, EXPECTED_COMMIT)
!ls data/sources/msc/images | wc -l

In [ ]:
!pip install -q cellpose==4.2.1.1 tifffile scikit-image opencv-python-headless pyyaml
import torch
print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Runtime > Change runtime type > GPU (A100)'
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Same model as the product: the Cellpose-SAM weights Colab downloads must hash to the ones every shipped
# record names (demo/examples/examples.json, model_weights_hash.seg).
from cellpose import models
_ = models.CellposeModel(gpu=True)
w = os.path.join(str(models.MODEL_DIR), 'cpsam_v2')
h = hashlib.sha256(open(w, 'rb').read()).hexdigest()
print('cpsam_v2', h)
assert h == '0f1cc3f7ecdd8a037a57c6c48d9d8921391be4cbce3fa9f13c3e3a2e1253c667', 'not the shipped weights: stop and tell me'
# The pre-registered files are byte-identical to the pre-registration commit
SEALED = {
 "scripts/confluency_mcellseg.py": "fd236fe2d51f8d1cd777450e4c6e257b6b44fa1a768428859c7f20c11df9f084",
 "results/confluency_mcellseg_split.json": "84f6fb7e4eb99bdefc5f07ce7aeace8f7d0bfb68ff8f4bd7422c8348ef029c15",
 "scripts/confluency_mcellseg_swap.py": "a04fc1887e24766b79236a62e00bb0fed0a50991b2c286af2c1f9c1b944a62e5",
 "results/confluency_mcellseg_swap.md": "c9bf39b206adad9c45d7ed0e277a75943fb54e224af778b00a60da611c9480be"
}
for f, s in SEALED.items():
    assert hashlib.sha256(open(f, 'rb').read()).hexdigest() == s, f
print('pre-registered files match', EXPECTED_COMMIT)

In [ ]:
# mCellSeg (Alam et al. 2026, CC BY 4.0) straight from Zenodo to local disk, MD5 from the record
t0 = time.time()
os.makedirs('/content/dl', exist_ok=True)
!curl -sL -o /content/dl/mCellSeg.zip "https://zenodo.org/api/records/20174259/files/mCellSeg.zip/content"
md5 = hashlib.md5(open('/content/dl/mCellSeg.zip', 'rb').read()).hexdigest()
print('md5', md5)
assert md5 == 'ed06d6e4c10e93b81703984cef852246'
os.makedirs('data/sources/mcellseg', exist_ok=True)
!unzip -q -o /content/dl/mCellSeg.zip -d data/sources/mcellseg
os.remove('/content/dl/mCellSeg.zip')
!ls data/sources/mcellseg/mCellSeg/labeled/images | wc -l
print(f'{time.time()-t0:.0f} s')

In [ ]:
# Masks only: the split must match the committed one
!python scripts/confluency_mcellseg.py count

### ⏸ Checkpoint
The last line above must say **`split matches the committed results/confluency_mcellseg_split.json`**, and the
cell before it **`pre-registered files match`**. If either fails, stop and send me the output.

In [ ]:
def save_results(label):
    # one small zip to Drive: reading file + run manifests + logs (no maps, no weights)
    keep = ['results/confluency_mcellseg_swap_curves.json']
    keep += [os.path.join(r, f) for r, _, fs in os.walk('cache/finetune') for f in fs if f == 'manifest.json']
    keep += [os.path.join('logs', f) for f in sorted(os.listdir('logs'))] if os.path.isdir('logs') else []
    local = '/content/confluency_v3_results.zip'
    with zipfile.ZipFile(local, 'w', zipfile.ZIP_DEFLATED) as z:
        z.writestr('COMMIT', commit)
        for p in keep:
            if os.path.exists(p):
                z.write(p)
    shutil.copy(local, RESULTS_ZIP)
    print(label, '->', RESULTS_ZIP, f'{os.path.getsize(local)/1e6:.1f} MB')
    with zipfile.ZipFile(local) as z:
        for n in z.namelist(): print('  ', n)

os.makedirs('logs', exist_ok=True)
def run(cmd, log):
    # streams the output (minus progress bars) and keeps a full log; stops the cell if the step fails
    t0 = time.time()
    r = subprocess.run(['bash', '-c', f'set -o pipefail; {cmd} 2>&1 | tee -a logs/{log}.log | (grep -v -E "it/s]|s/it]" || true)'])
    print(f'[{log}] {(time.time()-t0)/60:.1f} min, exit {r.returncode}')
    if r.returncode != 0:
        save_results(f'{log} FAILED (partial results)')
        raise RuntimeError(f'{log} failed — send me logs/{log}.log (it is in the results zip)')

## Part A — the swapped replication (~1 h on A100)

Nothing here prints a score; scoring happens once, later, from the reading file.

In [ ]:
run('python scripts/confluency_mcellseg_swap.py train', 'swap_train')
run('python scripts/confluency_mcellseg_swap.py ftmaps', 'swap_ftmaps')
run('python scripts/confluency_mcellseg_swap.py curves', 'swap_curves')
save_results('Part A')

## Part B — a fine-tuned model for the MSC setup (~5 min)

Trains on the 20 MSC images fixed in the pre-registration. The weights (about 1 GB) are copied to Drive on
their own; only the manifest (with the weights' SHA-256) goes in the results zip.

In [ ]:
run('python scripts/confluency_finetune.py final --n 20', 'msc_final_train')
man = json.load(open('cache/finetune/msc_all_n20_s0/manifest.json'))
print(man['n'], 'images; weights sha256', man['weights_sha256'])
shutil.copy(man['weights'], f'{STAGE_DIR}/msc_all_n20_s0.weights')
assert hashlib.sha256(open(f'{STAGE_DIR}/msc_all_n20_s0.weights', 'rb').read()).hexdigest() == man['weights_sha256']
print('weights on Drive:', f'{STAGE_DIR}/msc_all_n20_s0.weights')
save_results('Part B')

## Done
Download `MyDrive/cultureqc/staged/confluency_v3_results.zip` and tell me where it is (Downloads is fine).